# Phân Vùng Xe Cộ Bằng YOLO26n-seg

Notebook này chạy cùng `vehicle_segmentation.py` để predict, validate và so sánh YOLO26n-seg với YOLO11n-seg trên các lớp xe của COCO.

## 1. Cài đặt thư viện

Nếu chạy trên Colab, upload hoặc clone toàn bộ thư mục project trước khi chạy notebook.

In [ ]:
import pathlib, subprocess, sys

requirements = pathlib.Path('requirements.txt')
if requirements.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
else:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'ultralytics>=8.4.31', 'pycocotools>=2.0.7',
        'opencv-python>=4.9', 'pandas>=2.0', 'matplotlib>=3.8'
    ])

## 2. Kiểm tra môi trường

In [ ]:
!python vehicle_segmentation.py check-env

## 3. Sanity check với COCO128-Seg

Bước này tải dataset nhỏ, chạy predict 10 ảnh và validate nhanh trên 5 lớp xe.

In [ ]:
!python vehicle_segmentation.py sanity --model yolo26n-seg.pt --batch 4

## 4. Predict ảnh minh họa mask

Tạo danh sách ảnh có nhãn xe rồi predict 10 ảnh từ danh sách này.

In [ ]:
!python vehicle_segmentation.py build-subset \
  --source-root data/coco128-seg \
  --output-dir outputs/subsets/coco128_vehicle \
  --train-limit 30 \
  --val-limit 10

In [ ]:
!python vehicle_segmentation.py predict \
  --model yolo26n-seg.pt \
  --source outputs/subsets/coco128_vehicle/train.txt \
  --max-images 10 \
  --name vehicle_examples

## 5. So sánh YOLO26n-seg và YOLO11n-seg trên COCO128-Seg

Dùng bước này để lấy bảng metric nhanh. Với báo cáo chính, đổi `--data data/coco128-seg.yaml` thành `--data coco.yaml` nếu đủ thời gian và dung lượng.

In [ ]:
!python vehicle_segmentation.py compare \
  --data data/coco128-seg.yaml \
  --models yolo26n-seg.pt yolo11n-seg.pt \
  --batch 4

## 6. Xem bảng kết quả

In [ ]:
import pandas as pd

summary = pd.read_csv('outputs/compare/comparison_summary.csv')
summary

In [ ]:
per_class = pd.read_csv('outputs/compare/comparison_per_class.csv')
per_class

## 7. Đánh giá chính trên COCO val2017

Bước này chỉ tải COCO val2017 và nhãn segmentation, nhẹ hơn nhiều so với tải toàn bộ COCO train/test.

In [ ]:
# !python vehicle_segmentation.py prepare-coco-val
# !python vehicle_segmentation.py compare \
#   --data data/coco-val2017-seg.yaml \
#   --models yolo26n-seg.pt yolo11n-seg.pt \
#   --batch 8 \
#   --device auto